In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
!pip uninstall -y transformers peft accelerate protobuf
!pip uninstall -y tokenizers
!pip install -q protobuf==3.20.3

!pip install -q \
    transformers==4.41.0 \
    peft==0.10.0 \
    accelerate \
    datasets \
    evaluate \
    sentencepiece \
    nltk \
    bert-score


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Found existing installation: transformers 4.41.0
Uninstalling transformers-4.41.0:
  Successfully uninstalled transformers-4.41.0
Found existing installation: peft 0.10.0
Uninstalling peft-0.10.0:
  Successfully uninstalled peft-0.10.0
Found existing installation: accelerate 1.11.0
Uninstalling accelerate-1.11.0:
  Successfully uninstalled accelerate-1.11.0
Found existing installation: protobuf 3.20.3
Uninstalling protobuf-3.20.3:
  Successfully uninstalled protobuf-3.20.3


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Found existing installation: tokenizers 0.19.1
Uninstalling tokenizers-0.19.1:
  Successfully uninstalled tokenizers-0.19.1


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.3 which is incompatible.
pydrive2 1.21.3 requires p

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [8]:
!pip install -q accelerate==0.28.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.1/290.1 kB 9.8 MB/s eta 0:00:00


In [ ]:

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# FORCE a clean Accelerate state (critical fix)
from accelerate import Accelerator
acc = Accelerator()

# 1) INSTALL DEPENDENCIES
print("Installing metrics...")
!pip install -q nltk bert-score

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
print("✓ Metrics installed.\n")

# 2) IMPORTS
print("Importing modules...")

import numpy as np
import random
import torch
from datasets import load_dataset
import evaluate
from bert_score import score as bertscore_score

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

from tqdm.auto import tqdm
from transformers import TrainerCallback

print("✓ Imports loaded.\n")

# CUSTOM REAL-TIME LOGGING CALLBACK
class RealTimeLoggingCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, **kwargs):
        print(f"\n======== 🚀 Starting Epoch {state.epoch} ========\n")
    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"\n======== 🎉 Finished Epoch {state.epoch} ========\n")
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 20 == 0:
            last = state.log_history[-1] if state.log_history else {}
            loss = last.get("loss", "N/A")
            print(f"Step {state.global_step} | Loss: {loss}")

# 3) CONFIG
MODEL_NAME = "google/flan-t5-base"
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 64
BATCH_SIZE = 4
GRAD_ACCUM = 4
NUM_EPOCHS = 2
SEED = 42
OUTPUT_DIR = "/kaggle/working/flan_t5_qg"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("✓ Configuration loaded.\n")

# 4) LOAD DATA
print("Loading SQuAD v2...")
dataset = load_dataset("squad_v2")

train_split = dataset["train"].train_test_split(
    test_size=0.2, seed=SEED
)
train_dataset = train_split["train"]
val_dataset   = train_split["test"]
test_dataset  = dataset["validation"]

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}\n")

# 5) TOKENIZER & TOKENIZATION
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    inputs = ["generate question: " + c for c in examples["context"]]
    enc = tokenizer(inputs, padding="max_length", truncation=True,
                    max_length=MAX_SOURCE_LENGTH)

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["question"],
            padding="max_length",
            truncation=True,
            max_length=MAX_TARGET_LENGTH
        )

    enc["labels"] = labels["input_ids"]
    return enc

print("Tokenizing...")
train_tok = train_dataset.map(preprocess, batched=True, desc="Train tokenize")
val_tok   = val_dataset.map(preprocess, batched=True, desc="Val tokenize")
test_tok  = test_dataset.map(preprocess, batched=True, desc="Test tokenize")

print("\n✓ Tokenization complete.\n")


# 6) LOAD MODEL

print("Loading FLAN-T5-base model...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
collator = DataCollatorForSeq2Seq(tokenizer, model=model)
print("✓ Model loaded.\n")


# 7) METRICS

print("Loading metrics...")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")
print("✓ Metrics ready.\n")

def compute_qg_metrics(preds, refs):
    preds = [p.strip() for p in preds]
    refs  = [r.strip() for r in refs]

    print("Computing final metrics...")

    rouge_out  = rouge.compute(predictions=preds, references=refs)
    meteor_out = meteor.compute(predictions=preds, references=refs)

    _, _, F1 = bertscore_score(preds, refs, model_type="bert-base-uncased", lang="en")
    return {
        "rouge1": rouge_out["rouge1"],
        "rouge2": rouge_out["rouge2"],
        "rougeL": rouge_out["rougeL"],
        "meteor": meteor_out["meteor"],
        "bertscore_f1": float(F1.mean())
    }

def compute_metrics(eval_pred):
    ids, labels = eval_pred
    if isinstance(ids, tuple):
        ids = ids[0]

    preds = tokenizer.batch_decode(ids, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    refs = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return compute_qg_metrics(preds, refs)


# 8) TRAINING ARGS (NO FREEZES)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=20,

    # KAGGLE-FRIENDLY (NO FP16 FREEZES)
    bf16=True,

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,

    dataloader_num_workers=2,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[RealTimeLoggingCallback()],
)


print("\n Training is starting NOW...\n")
trainer.train()
print("\n TRAINING COMPLETE!\n")

# ================================================================
# 10) SAVE MODEL
# ================================================================
print("Saving model...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✓ Model saved.\n")

# ================================================================
# 11) TEST SET GENERATION
# ================================================================
print("Generating questions for test set...")

def generate_for_dataset(dataset, batch_size=8):
    preds, refs = [], []
    for i in tqdm(range(0, len(dataset), batch_size), desc="Generating"):
        batch = dataset[i:i+batch_size]
        inputs = ["generate question: " + c for c in batch["context"]]

        enc = tokenizer(inputs, return_tensors="pt",
                        padding=True, truncation=True,
                        max_length=MAX_SOURCE_LENGTH).to(model.device)
        out = model.generate(**enc, num_beams=4, max_length=64)
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
        refs.extend(batch["question"])
    return preds, refs

preds, refs = generate_for_dataset(test_dataset)

# ================================================================
# 12) FINAL TEST METRICS
# ================================================================
print("Computing test metrics...")
test_metrics = compute_qg_metrics(preds, refs)

print("\n===== FINAL TEST METRICS =====")
for k, v in test_metrics.items():
    print(f"{k}: {v}")

# ================================================================
# 13) SAMPLES
# ================================================================
print("\nShowing samples:\n")
for i in range(5):
    print("\n------------------------------------------------")
    print("Context:", test_dataset[i]["context"][:300], "...")
    print("Actual :", refs[i])
    print("Pred   :", preds[i])


Installing metrics...
✓ Metrics installed.

Importing modules...
✓ Imports loaded.

✓ Configuration loaded.

Loading SQuAD v2...


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Train: 104255 | Val: 26064 | Test: 11873

Loading tokenizer...
Tokenizing...


Train tokenize:   0%|          | 0/104255 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3946: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


...

Test tokenize:   0%|          | 0/11873 [00:00<?, ? examples/s]


✓ Tokenization complete.

Loading FLAN-T5-base model...
✓ Model loaded.

Loading metrics...


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


✓ Metrics ready.


🚀 Training is starting NOW...



<IPython.core.display.Javascript object>